In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat
import pickle

from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques
)
from scipy.io import loadmat

probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y
probe_position['chan_map'] = probe_data['chanMap0ind'].astype(int)

chan_map = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
merged = chan_map.merge(probe_position, left_on='probeloc', right_on='chan_map')\
                 .iloc[chan_map.index]\
                 .reset_index(drop=True)

probe = Probe()
probe.set_contacts(positions=merged.iloc[:, 2:4])
probe.set_device_channel_indices(range(256))

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
recording_raw = se.read_intan(f"/home/ubuntu/Documents/jct/project/251205/M190011_260121_150111_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)

print('read success')

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_raw = spre.resample(recording_raw, 10000)

recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

rec_params_raw = pd.read_csv("/media/ubuntu/sda/duan/result/260121/rec_params.csv")
rec_params_raw = rec_params_raw[rec_params_raw['bhv_codes'] == 10]

original_fs = 30000
target_fs = 10000
fs_ratio = original_fs / target_fs
rec_params_raw['rec_codes_points_10000'] = (rec_params_raw['rec_codes_points'] / fs_ratio).astype(int)

trials_per_group = 1000
num_groups_to_process = 5
buffer_seconds = 10
fs = recording_f.get_sampling_frequency()
buffer_samples = int(buffer_seconds * fs)

first_group_min_sample = None
last_group_max_sample = None

for group_idx in range(num_groups_to_process):
    trial_start = group_idx * trials_per_group + 1
    trial_end = (group_idx + 1) * trials_per_group
    group_params = rec_params_raw[(rec_params_raw['trial_ids'] >= trial_start) & (rec_params_raw['trial_ids'] <= trial_end)]
    if len(group_params) == 0:
        continue

    min_sample = int(group_params['rec_codes_points_10000'].min())
    max_sample = int(group_params['rec_codes_points_10000'].max())
    if group_idx == 0:
        first_group_min_sample = min_sample
    if group_idx == num_groups_to_process - 1:
        last_group_max_sample = max_sample

start_sample = max(0, first_group_min_sample - buffer_samples)
end_sample = min(recording_f.get_num_samples(), last_group_max_sample + buffer_samples)
print(f"Slicing recording for groups 1-{num_groups_to_process}: {start_sample} to {end_sample} samples")

recording_segment = recording_f.frame_slice(start_frame=start_sample, end_frame=end_sample)


read success
Slicing recording for groups 1-5: 0 to 36770347 samples


In [3]:
from utils_clique import plot_cliques


output_folder = '/media/ubuntu/sda/mouse_test/sorted/end2end_neuroscroll'
cliques = build_sliding_cliques(
    probe,
    clique_size=32,
    min_size=25,
    min_overlap=6,
    target_groups=10,
)

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 49,
        'min_size': 25,
        'min_overlap': 6,
        'target_groups': 10,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

plot_cliques(probe, cliques, f'{output_folder}/cliques.pdf')

[INFO] Built 10 cliques (target 10)
       Clique 00: channels 204-39 (32 channels)
       Clique 01: channels 74-81 (32 channels)
       Clique 02: channels 174-227 (32 channels)
       Clique 03: channels 33-88 (32 channels)
       Clique 04: channels 69-253 (32 channels)
       Clique 05: channels 229-232 (32 channels)
       Clique 06: channels 35-24 (32 channels)
       Clique 07: channels 207-141 (32 channels)
       Clique 08: channels 134-109 (32 channels)
       Clique 09: channels 148-115 (32 channels)

Clique可视化PDF已保存至: /media/ubuntu/sda/mouse_test/sorted/end2end_neuroscroll/cliques.pdf


In [ ]:

print(f"\n{'='*60}")
print(f"开始训练模型：每个clique使用segment_0数据训练5次")
print(f"{'='*60}\n")

# 对每个clique进行训练
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"训练 Clique {clique_id}")
    print(f"{'='*60}")
    
    neuron_inf_path = f'/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/neuron_inf.pickle'
    gt_detect_array_path = f'/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/gt_detect_array.csv'
    

    with open(neuron_inf_path, 'rb') as f:
        neuron_inf_dict = pickle.load(f)
    gt_detect_array = pd.read_csv(gt_detect_array_path)
    
    neuron_inf_segment = neuron_inf_dict_to_dataframe(neuron_inf_dict)

    recording_clique = get_recording_clique(recording_segment, clique)
    print(f"  Recording clique channels: {len(recording_clique.get_channel_ids())}")
    
    # 保持gt_detect_array的time为采样点索引（不转换为秒）
    gt_detect_array_for_training = gt_detect_array.copy()
    
    # 将extremum_channel转换为字符串类型，以匹配recording_clique的channel IDs
    if 'extremum_channel' in gt_detect_array_for_training.columns:
        gt_detect_array_for_training['extremum_channel'] = gt_detect_array_for_training['extremum_channel'].astype(str)
    
    gt_detect_array_for_training.columns = ['time', 'unit_id', '']
    # 准备训练数据
    clique_save_dir = f'{output_folder}/clique_{clique_id}/'
    train_data_dir = prepare_training_data(
        recording_f=recording_clique,
        gt_detect_array=gt_detect_array_for_training,
        neuron_inf=neuron_inf_segment,
        save_dir=clique_save_dir,
        duration_seconds=100, 
        thr_min=2.5,
        thr_max=30,
        distance=3,
        wlen=5,
        prominence=15,
        left_sample=10,
        right_sample=20,
        max_firing_channel=None
    )
    
    n_channels = recording_clique.get_num_channels()
    n_repeats = 1
    
    for repeat_idx in range(1, n_repeats + 1):
        print(f"\n  ===== 重复训练 {repeat_idx}/{n_repeats} =====")
        model_save_dir = f'{clique_save_dir}/model_{repeat_idx}'
        
        autosort_model, training_log = train_autosort_model(
            train_data_dir=train_data_dir,
            model_save_dir=model_save_dir,
            n_channels=n_channels,
            left_sample=10,
            right_sample=20,
            epochs=20,
            batch_size=512,
            device=None,
            early_stopping=True,
            patience=5,
            min_delta=0.0,
            use_focal_loss=True,
            focal_gamma=2.0
        )
        
        print(f"  重复训练 {repeat_idx}/{n_repeats} 完成!")
    
    print(f"  Clique {clique_id} 所有重复训练完成!")

print("\n所有训练完成！")



开始训练模型：每个clique使用segment_0数据训练5次


训练 Clique 0


FileNotFoundError: [Errno 2] No such file or directory: '/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/neuron_inf.pickle'